In [7]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))
from quantlab.data.binance import load_parquet
from quantlab.features.technical_features import add_returns, add_momentum, add_sma
from quantlab.portfolio.allocation import portfolio_returns
from quantlab.backtest.strategy import sign_signal, net_returns
from quantlab.backtest.report import performance_table, format_table

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
df = load_parquet('../data/raw/BTCUSDT_1d.parquet')
df = add_returns(df)
df = add_momentum(df, 20)
df = add_sma(df, 20)

In [3]:
# Signals: +1 (long), -1 (short), 0 while the feature is still undefined
momentum_20d_signal = sign_signal(df['momentum_20'])
momentum_moving_average_signal = sign_signal(df['close'] / df['sma_20'] - 1)

# Strategy returns net of costs (10 bp per unit of position traded)
COST = 0.001
momentum_20d_return_net = net_returns(momentum_20d_signal, df['returns'], cost=COST)
momentum_moving_average_return_net = net_returns(momentum_moving_average_signal, df['returns'], cost=COST)

In [4]:
strategy_returns = pd.concat([momentum_20d_return_net, momentum_moving_average_return_net], axis=1).dropna()
strategy_returns.columns = ['momentum_20d', 'moving_average']

weights = pd.Series(data=[0.5, 0.5], index=strategy_returns.columns)

# equal-weight blend of the two rules
blend_returns = portfolio_returns(strategy_returns, weights)

correlation = strategy_returns.corr().iloc[0, 1]
print('Correlation between the two rules: {:.2f}'.format(correlation))

Correlation between the two rules: 0.68


In [5]:
table = performance_table({
    'buy_and_hold': df['returns'],
    'momentum_20d': momentum_20d_return_net,
    'moving_average': momentum_moving_average_return_net,
    'equal_weight_blend': blend_returns,
})
format_table(table)

,total_return,cagr,ann_volatility,sharpe,sharpe_se,var_95,exp_shortfall_95,max_drawdown
buy_and_hold,25.1%,8.5%,47.0%,+0.41,0.60,3.7%,5.3%,-53.0%
momentum_20d,-42.0%,-18.0%,45.8%,-0.21,0.60,3.8%,5.4%,-65.6%
moving_average,-21.4%,-8.4%,45.9%,+0.04,0.60,3.7%,5.5%,-54.6%
equal_weight_blend,-29.2%,-11.9%,41.9%,-0.09,0.60,3.5%,5.2%,-55.9%


### Conclusions

In [6]:
# Printed from the numbers above, so the text always matches them.
vol = table['ann_volatility']
print('- The two rules trade the same asset on similar information: their daily returns have correlation {:.2f}.'.format(correlation))
print('- The equal-weight blend therefore diversifies little: volatility {:.0%}, against {:.0%} and {:.0%} for the two rules.'.format(
    vol['equal_weight_blend'], vol['momentum_20d'], vol['moving_average']))
print('- This is a blend of two signals on one asset, not a portfolio. Diversification needs several assets,')
print('  or signals with low correlation, and weights chosen from an estimate of their covariance.')

- The two rules trade the same asset on similar information: their daily returns have correlation 0.68.
- The equal-weight blend therefore diversifies little: volatility 42%, against 46% and 46% for the two rules.
- This is a blend of two signals on one asset, not a portfolio. Diversification needs several assets,
  or signals with low correlation, and weights chosen from an estimate of their covariance.
